<a href="https://colab.research.google.com/github/ericb42/GB885-Final-Brauer-E/blob/main/GB885_Final_Brauer_E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Project for GB885 - Python Fundamentals

# RUSH Case Study
RUSH is a globally renowned sportswear and footwear brand.

This notebook is meant to provide analysis of RUSH sales data for use in understanding the market and identifying opportunities for growth.

In [ ]:
# install external dependencies

In [ ]:
# import modules and packages
import pandas as pd

In [ ]:
# load data files
products_url = 'https://raw.githubusercontent.com/ericb42/GB885-Final-Brauer-E/refs/heads/main/data/TABLE_PRODUCTS_885.csv'
retailer_url = 'https://raw.githubusercontent.com/ericb42/GB885-Final-Brauer-E/refs/heads/main/data/TABLE_RETAILER_885.csv'
sales_url = 'https://raw.githubusercontent.com/ericb42/GB885-Final-Brauer-E/refs/heads/main/data/TABLE_SALES_885.csv'

# products data (pipe separated)
products_df = pd.read_csv(products_url, sep = '|')

# retailer data (comma separated)
retailer_df = pd.read_csv(retailer_url)

# sales data (comma separated)
sales_df = pd.read_csv(sales_url)

## Preview the data sets

In [ ]:
# preview products_df
products_df.head()

In [ ]:
products_df.info()

In [ ]:
# preview retailer_df
retailer_df.head()

In [ ]:
retailer_df.info()

In [ ]:
# preview sales_df
sales_df.head()

In [ ]:
sales_df.info()

## Initial clean up of data sets
When originally consolidating the dataframes, we found that the number of records increased from 9,648 to 10,271. This occurred because the RETAILER_ID column in retailer_df, which is identified as the primary key, contains duplicate values. When a sales record matches one of these duplicated RETAILER_ID values, the merge produces multiple rows for that sale.  

Because there is no reliable way to determine which retailer a duplicated RETAILER_ID should reference, we will not attempt to assign those sales to a specific retailer. However, since the business questions focus primarily on state-level analysis, we can preserve the validity of those analyses by removing only the retailer information for the affected records while retaining their state and city information.  

One duplicated RETAILER_ID presents the opposite situation: the retailer information is distinguishable, but the state and city information are identical and therefore cannot be uniquely assigned. For this case, we will remove one of the duplicate RETAILER_ID records and clear the STATE and CITY values from the remaining record. This preserves the retailer information while preventing incorrect geographic attribution.  

This approach maximizes the amount of usable data while avoiding assumptions about which retailer or geographic location should be associated with any ambiguous sales record.

In [ ]:
# identify retailer_df duplicates

retailer_df[retailer_df.duplicated(subset=['RETAILER_ID'], keep=False)].sort_values('RETAILER_ID')

In [ ]:
# create list of duplicate indices
duplicate_indicies = [63,81,84,83]

# drop records for duplicate_indicies
retailer_dedup_df = retailer_df.drop(duplicate_indicies)

# check our work
retailer_dedup_df.info()
retailer_dedup_df[retailer_dedup_df.duplicated(subset=['RETAILER_ID'], keep=False)].sort_values('RETAILER_ID')

In [ ]:
# remove the RETAILER info for the applicable ambiguous RETAILER_ID records
amb_retailer_id = ['W00SARLI', 'W00SFLOR', 'W00STEHO']
retailer_dedup_df.loc[retailer_dedup_df['RETAILER_ID'].isin(amb_retailer_id), ['RETAILER']] = ''

# check our work
retailer_dedup_df.loc[retailer_dedup_df['RETAILER_ID'].isin(amb_retailer_id)]

In [ ]:
# remove STATE and CITY info for the applicable ambiguous RETAILER_ID records
amb_retailer_id = ['S00NNENE']
retailer_dedup_df.loc[retailer_dedup_df['RETAILER_ID'].isin(amb_retailer_id), ['STATE', 'CITY']] = ''

# check our work
retailer_dedup_df.loc[retailer_dedup_df['RETAILER_ID'].isin(amb_retailer_id)]

## Consolidate the dataframes

In [ ]:
# merge sales data and retailer data based on RETAILER_ID
sales_retailer_df = pd.merge(sales_df, retailer_dedup_df, on = 'RETAILER_ID', how = 'left')

# check our work
sales_retailer_df.head()

In [ ]:
sales_retailer_df.info()

In [ ]:
# merge new sales_retailer_df with products_df using PRODUCT_ID
sales_retailer_products_df = pd.merge(sales_retailer_df, products_df, on = 'PRODUCT_ID', how = 'left')

# create srp_df reference to sales_retailer_products_df for more concise coding
srp_df = sales_retailer_products_df

# check our work
srp_df.head()

In [ ]:
srp_df.info()

## Inspect the Data

### Check for null values

In [ ]:
# Traditional null values
srp_df.isnull().sum()

Findings:
 - (2) null values in PRICE_PER_UNIT
 - (1) null value in RETAILER
 - (1) null value in REGION
 - (1) null value in STATE
 - (1) null value in CITY

In [ ]:
# non-traditional null values: categorical data
# list of categorical variables in dataframe
cat_var = list(srp_df.select_dtypes(include = ['object']).columns)

for column in cat_var:
  print(column)
  print(srp_df[column].unique())

In [ ]:
# list of features to convert to numeric
to_numeric = [ 'UNITS_SOLD']

# unique vlues in features expected to be numeric
for column in to_numeric:
  print(column)
  print(srp_df[column].unique())

Findings:
 - There is a '999999999' in RETAILER_ID
 - There is a '***' in UNITS_SOLD

In [ ]:
# non-traditional null values: numerical data
srp_df.describe()

Findings:
 - 99999.000000 in PRICE_PER_UNIT




### Check for duplicate values

In [ ]:
# check for duplicates
srp_df.duplicated().sum()

In [ ]:
srp_df.to_csv('srp_df.csv')
